## Imports

In [3]:
# %pip install matplotlib tensorflow keras mnist scikit-learn pandas opencv-python opencv-contrib-python

In [4]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import image_dataset_from_directory

# helper libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import pandas as pd
from PIL import Image
import os

## Load data

In [5]:
# version with numpy array's for easier viewing of data
train_data = np.genfromtxt('train.csv',delimiter=',',skip_header=1, dtype='str')
train_x = train_data[:,1]
train_y = train_data[:,0]
X_train,  X_test, y_train, y_test = train_test_split(train_x,train_y,test_size=0.2)

# get image data via tf.keras.utils.image_dataset_from_directory. Folder structure is made for it. This prefents problems with only having the image name and not the pixel values
train_datatset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    batch_size=50,
    validation_split=0.2,
    subset="training",
    seed=42
)

test_dataset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    batch_size=50,
    validation_split=0.2,
    subset="validation",
    seed=42
)

Found 1334 files belonging to 7 classes.
Using 1068 files for training.
Found 1334 files belonging to 7 classes.
Using 266 files for validation.


## Check labels

<table>
<tr><th>Value</th><th>Class</th></tr>
<tr><td>0</td>	<td>Black_spiny_tailed_iguana</td></tr>
<tr><td>1</td>	<td>Brown_anole</td></tr>
<tr><td>2</td>	<td>Cuban_knight_anole</td></tr>
<tr><td>3</td>	<td>Desert_iguana</td></tr>
<tr><td>4</td>	<td>Green_anole</td></tr>
<tr><td>5</td>	<td>Green_iguana</td></tr>
<tr><td>6</td>	<td>Lesser_Antillean_iguana</td></tr>
</table>


In [6]:
y_test_unique, y_test_count = np.unique(y_test,return_counts=True)
y_train_unique, y_train_count = np.unique(y_train,return_counts=True)
print(y_test_unique)
print(y_test_count)
print(y_train_unique)
print(y_train_count)

['Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_10.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_102.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_116.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_117.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_12.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_123.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_124.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_14.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_141.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_143.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_157.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_160.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_171.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_176.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_183.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_192.

To check if the label distribution is random we print the first and last of test and train data.

In [7]:
# Print the first and last 10 labels from train/test set
print(y_test[:10])
print(y_test[-10:])
print(y_train[:10])
print(y_train[-10:])

['Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_53.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_141.jpg'
 'Desert_iguana_Desert_iguana_18.jpg' 'Brown_anole_Brown_anole_116.jpg'
 'Brown_anole_Brown_anole_168.jpg'
 'Cuban_knight_anole_Cuban_knight_anole_150.jpg'
 'Brown_anole_Brown_anole_57.jpg' 'Brown_anole_Brown_anole_112.jpg'
 'Green_anole_Green_anole_36.jpg' 'Green_iguana_Green_iguana_119.jpg']
['Cuban_knight_anole_Cuban_knight_anole_189.jpg'
 'Lesser_Antillean_iguana_Lesser_Antillean_iguana_18.jpg'
 'Desert_iguana_Desert_iguana_150.jpg' 'Brown_anole_Brown_anole_192.jpg'
 'Green_iguana_Iguana_iguana_7.jpg' 'Desert_iguana_Desert_iguana_153.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_176.jpg'
 'Lesser_Antillean_iguana_Lesser_Antillean_iguana_26.jpg'
 'Cuban_knight_anole_Cuban_knight_anole_10.jpg'
 'Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_25.jpg']
['Black_spiny-tailed_iguana_Black_spiny-tailed_iguana_199.jpg'
 'Green_anole_Green_anole_125.j

Here we make a grid of the first 10 images to get an overview of what images we are working with.

In [8]:
# plt.figure(figsize=(10,10))
# for i in range(10):
#     plt.subplot(4,5,i+1)
#     plt.xticks([])
#     plt.yticks([])
#     plt.grid(False)
#     plt.imshow(X_train[i],cmap='binary')
    # plt.xlabel(class_names[y_train[i]])

## Define model settings

In [9]:
# Define your sequential model, with the different layers, including the preprocessing layers
model =keras.Sequential([
    # Preprocessing: Add a Rescaling layer to rescale the pixel values to the [0, 1] range
    layers.Rescaling(1./255),
    # Input Layer: Add a Flattening layer to make 1-D vector of our 28x28 images
    layers.Flatten(input_shape=(28, 28)),
    # Hidden layer: Add a Dense layer, aka a fully or densely connected layer of 128 neurons, and let them use the 'ReLu'-squishing or activation function
    layers.Dense(128, activation='relu'),
    
    # Extra hidden layers
    layers.Dense(512, activation='relu'),

    # dropout layer applies to the layer above it 
    # dropout disabels the selected amount(25%) of neurons during one run of the model to decrease the reliance on specific neurons
    keras.layers.Dropout(0.25), 
    layers.Dense(512, activation='relu'),
    # dropout layer applies to the layer above it
    keras.layers.Dropout(0.25),  
    layers.Dense(256, activation='sigmoid'),
    layers.Dense(128, activation='sigmoid'),
    
    # Output layer: Add a Dense layer of 10 neurons (because we have 10 possible output labels), and link those neurons together in a group, via the 'softmax'-activation function
    layers.Dense(10, activation='softmax')
]) 

c:\Artificial inteligence - DL\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Compile model

In [10]:
# Compile the model with the adam optimizer with lr of 0.001, sparse categorical loss, and accuracy as metric
model.compile(loss='sparse_categorical_crossentropy',
              optimizer=optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])

## Train model

In [11]:
# Train the model for 20 epochs, in batches of 50, and use a validation split of 20%.
history = model.fit(train_datatset,batch_size=50,epochs=20,)

Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 253ms/step - accuracy: 0.1199 - loss: 2.0757
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.1217 - loss: 1.9806
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 238ms/step - accuracy: 0.1498 - loss: 1.9687
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 239ms/step - accuracy: 0.1414 - loss: 1.9591
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 243ms/step - accuracy: 0.1367 - loss: 1.9653
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 256ms/step - accuracy: 0.1414 - loss: 1.9594
Epoch 7/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 250ms/step - accuracy: 0.1376 - loss: 1.9640
Epoch 8/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 249ms/step - accuracy: 0.1320 - loss: 1.9601
Epoch 9/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 256ms/step - accuracy: 0.1245 - loss: 1.9571
Epoch 10/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 247ms/step - accuracy: 0.1442 - loss: 1.9567
Epoch 11/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 243ms/step - accuracy: 0.1442 - loss: 1.9580
Epoch 12/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 248ms/step

## Validation

In [12]:
# Plot the training process

def plot_training(history):
    # put the code here
    # Create a figure and a grid of subplots with a single call
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5))

    # Plot the loss curves on the first subplot
    ax1.plot(history.history['loss'], label='training loss')
    ax1.plot(history.history['val_loss'], label='validation loss')
    ax1.set_title('Loss curves')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Plot the accuracy curves on the second subplot
    ax2.plot(history.history['accuracy'], label='training accuracy')
    ax2.plot(history.history['val_accuracy'], label='validation accuracy')
    ax2.set_title('Accuracy curves')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    # Adjust the spacing between subplots
    fig.tight_layout()

    # Show the figure
plt.show()

In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 196608)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    25,165,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,980,512 (293.66 MB)

 Trainable params: 25,660,170 (97.89 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 51,320,342 (195.77 MB)

## Prediction

In [14]:
test_loss, test_acc = model.evaluate(test_dataset)

print('Test accuracy:', test_acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step - accuracy: 0.1353 - loss: 1.9519
Test accuracy: 0.13533835113048553


In [15]:
# First, let's generate the predictions for all the test images
predictions = model.predict(train_datatset)
# Next, we'll transform all the prediction into the winners (otherwise each prediction gives us the 10 probabilities, but we only need the winner, the one our network thinks it is)
pred = np.argmax(predictions, axis=1)


22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step


In [16]:
import pandas as pd

df = pd.DataFrame(data={"TARGET": pred.flatten()})
df.index.name = "ID" 
df.to_csv("submission.csv", index=True)